# Train NN1

Builds the site bundle with `plotluck.build`: 5-fold district-grouped CV of NN1 against a training-mean baseline, then the shipped model on every row for the median best epoch. On full data, the run results go to `runs/<yyyy-mm-dd>-train-nn1/` to commit; on sample data, to the sample's `run/` folder. The bundle and the private `restaurants.csv` and `label_policy.json` go to the data folder.

In [ ]:
import json
from datetime import date

import matplotlib.pyplot as plt

from plotluck import build, data_dir

data = data_dir()
full = data == data_dir(choice="full")
run = data.parents[1] / "runs" / f"{date.today():%Y-%m-%d}-train-nn1" if full else data / "run"
print(data, "->", run)

In [ ]:
build.main(["--run", str(run)])
metrics = json.loads((run / "metrics.json").read_text(encoding="utf-8"))
cv = metrics["cv"]

In [ ]:
print(f"{'fold':>4} {'best epoch':>10} {'MAE':>7} {'RMSE':>7} {'baseline MAE':>12} {'baseline RMSE':>13}")
for i, f in enumerate(cv["folds"], 1):
    print(f"{i:>4} {f['best_epoch']:>10} {f['mae']:>7.4f} {f['rmse']:>7.4f} {f['baseline_mae']:>12.4f} {f['baseline_rmse']:>13.4f}")
for name, s in cv["summary"].items():
    print(f"{name:>13}: {s['mean']:.4f} ± {s['std']:.4f} (fold mean ± spread)")
print("shipped model epochs (median best epoch):", metrics["shipped"]["epochs"])

In [ ]:
# Loss per epoch, one panel per fold; saved to the run folder for the evidence checklist.
folds = cv["folds"]
fig, axes = plt.subplots(1, len(folds), figsize=(3.2 * len(folds), 3), sharey=True, facecolor="#fcfcfb")
for i, (ax, f) in enumerate(zip(axes, folds), 1):
    epochs = range(1, len(f["train_loss"]) + 1)
    ax.plot(epochs, f["train_loss"], color="#2a78d6", lw=2, label="training rows")
    ax.plot(epochs, f["early_stop_loss"], color="#eb6834", lw=2, label="early-stop slice")
    ax.axvline(f["best_epoch"], color="#52514e", lw=1, ls=":", label="best epoch")
    ax.set_title(f"Fold {i}", color="#0b0b0b")
    ax.set_xlabel("epoch", color="#52514e")
    ax.set_facecolor("#fcfcfb")
    ax.grid(axis="y", color="#e4e3df", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors="#52514e")
axes[0].set_ylabel("MSE", color="#52514e")
axes[0].legend(frameon=False)
fig.suptitle("NN1 loss per fold", color="#0b0b0b")
fig.tight_layout()
fig.savefig(run / "loss_curves.png", dpi=120)
plt.show()